In [0]:
# Databricks Authentication Done below
# Storage account name
storage_account = "smartgearstorage428"

# Azure AD credentials
client_id = "88f1979f-43c0-4853-b988-45f796c7726d"
tenant_id = "09cef3a2-844b-4ba2-80ba-22179ec4d11b"
client_secret = "C-58Q~BCmw9cDKCW6w~4rDylifrc5FbVlLTzea.e"

spark.conf.set(
f"fs.azure.account.auth.type.{storage_account}.dfs.core.windows.net",
"OAuth")

spark.conf.set(
f"fs.azure.account.oauth.provider.type.{storage_account}.dfs.core.windows.net",
"org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")

spark.conf.set(
f"fs.azure.account.oauth2.client.id.{storage_account}.dfs.core.windows.net",
client_id)

spark.conf.set(
f"fs.azure.account.oauth2.client.secret.{storage_account}.dfs.core.windows.net",
client_secret)

spark.conf.set(
f"fs.azure.account.oauth2.client.endpoint.{storage_account}.dfs.core.windows.net",
f"https://login.microsoftonline.com/{tenant_id}/oauth2/token")

In [0]:
# Test Storage Access
dbutils.fs.ls("abfss://raw@smartgearstorage428.dfs.core.windows.net/")

[FileInfo(path='abfss://raw@smartgearstorage428.dfs.core.windows.net/smartgear_sales.csv', name='smartgear_sales.csv', size=43437, modificationTime=1773553134000)]

In [0]:
# Explicit Schema
from pyspark.sql.types import *

schema = StructType([
StructField("OrderID", IntegerType(), True),
StructField("OrderDate", StringType(), True),
StructField("Region", StringType(), True),
StructField("StoreID", StringType(), True),
StructField("Product", StringType(), True),
StructField("Quantity", IntegerType(), True),
StructField("UnitPrice", DoubleType(), True)
])

In [0]:
# Read the  raw dataset using explicitW Schema
df = spark.read \
.format("csv") \
.option("header","true") \
.schema(schema) \
.load("abfss://raw@smartgearstorage428.dfs.core.windows.net/smartgear_sales.csv")

display(df)

OrderID,OrderDate,Region,StoreID,Product,Quantity,UnitPrice
1001,2025-03-14,East,115,Headphones,3,81.65
1002,2025-02-19,West,110,Smartwatch,3,242.5
1003,2025-03-17,West,113,Printer,2,152.59
1004,2025-01-05,West,118,Camera,4,463.4
1005,2025-01-07,West,110,Tablet,3,370.01
1006,2025-03-22,East,113,Smartphone,4,639.47
1007,2025-02-19,North,118,Drone,4,747.21
1008,2025-02-21,West,112,Printer,2,151.06
1009,2025-03-30,North,117,Smartphone,1,614.89
1010,2025-01-30,West,108,Tablet,3,415.43


In [0]:
# Prepare Bronze Dataset  by ingestion metadata columns

from pyspark.sql.functions import col, year, month, current_timestamp

bronze_df = df \
.withColumn("OrderDate", col("OrderDate").cast("date")) \
.withColumn("year", year("OrderDate")) \
.withColumn("month", month("OrderDate")) \
.withColumn("ingestion_time", current_timestamp())

display(bronze_df)

OrderID,OrderDate,Region,StoreID,Product,Quantity,UnitPrice,year,month,ingestion_time
1001,2025-03-14,East,115,Headphones,3,81.65,2025,3,2026-03-15T07:31:24.110021Z
1002,2025-02-19,West,110,Smartwatch,3,242.5,2025,2,2026-03-15T07:31:24.110021Z
1003,2025-03-17,West,113,Printer,2,152.59,2025,3,2026-03-15T07:31:24.110021Z
1004,2025-01-05,West,118,Camera,4,463.4,2025,1,2026-03-15T07:31:24.110021Z
1005,2025-01-07,West,110,Tablet,3,370.01,2025,1,2026-03-15T07:31:24.110021Z
1006,2025-03-22,East,113,Smartphone,4,639.47,2025,3,2026-03-15T07:31:24.110021Z
1007,2025-02-19,North,118,Drone,4,747.21,2025,2,2026-03-15T07:31:24.110021Z
1008,2025-02-21,West,112,Printer,2,151.06,2025,2,2026-03-15T07:31:24.110021Z
1009,2025-03-30,North,117,Smartphone,1,614.89,2025,3,2026-03-15T07:31:24.110021Z
1010,2025-01-30,West,108,Tablet,3,415.43,2025,1,2026-03-15T07:31:24.110021Z


In [0]:
# Creating Bronze folder with partitioning of year and month folders
bronze_df.write \
.mode("overwrite") \
.partitionBy("year","month") \
.parquet("abfss://bronze@smartgearstorage428.dfs.core.windows.net/smartgear_sales/")

In [0]:
# Data profiling on Bronze dataset
# Checks for null values and invalid numeric values
# (Quantity ≤ 0, UnitPrice ≤ 0) before Silver transformation
from pyspark.sql.functions import count, when

display(
    bronze_df.select(
        *[count(when(col(c).isNull(), c)).alias(f"{c}_nulls") for c in bronze_df.columns],
        count(when(col("Quantity") <= 0, True)).alias("invalid_quantity"),
        count(when(col("UnitPrice") <= 0, True)).alias("invalid_unitprice")
    )
)

OrderID_nulls,OrderDate_nulls,Region_nulls,StoreID_nulls,Product_nulls,Quantity_nulls,UnitPrice_nulls,year_nulls,month_nulls,ingestion_time_nulls,invalid_quantity,invalid_unitprice
0,0,0,0,0,0,0,0,0,0,0,0


In [0]:
# Silver transformation
# Remove invalid records, standardize Region values,
# and create Revenue = Quantity * UnitPrice

from pyspark.sql.functions import col, initcap

silver_df = bronze_df \
.filter((col("Quantity") > 0) & (col("UnitPrice") > 0)) \
.withColumn("Region", initcap(col("Region"))) \
.withColumn("Revenue", col("Quantity") * col("UnitPrice"))

display(silver_df)


OrderID,OrderDate,Region,StoreID,Product,Quantity,UnitPrice,year,month,ingestion_time,Revenue
1001,2025-03-14,East,115,Headphones,3,81.65,2025,3,2026-03-15T07:33:09.579975Z,244.95000000000002
1002,2025-02-19,West,110,Smartwatch,3,242.5,2025,2,2026-03-15T07:33:09.579975Z,727.5
1003,2025-03-17,West,113,Printer,2,152.59,2025,3,2026-03-15T07:33:09.579975Z,305.18
1004,2025-01-05,West,118,Camera,4,463.4,2025,1,2026-03-15T07:33:09.579975Z,1853.6
1005,2025-01-07,West,110,Tablet,3,370.01,2025,1,2026-03-15T07:33:09.579975Z,1110.03
1006,2025-03-22,East,113,Smartphone,4,639.47,2025,3,2026-03-15T07:33:09.579975Z,2557.88
1007,2025-02-19,North,118,Drone,4,747.21,2025,2,2026-03-15T07:33:09.579975Z,2988.84
1008,2025-02-21,West,112,Printer,2,151.06,2025,2,2026-03-15T07:33:09.579975Z,302.12
1009,2025-03-30,North,117,Smartphone,1,614.89,2025,3,2026-03-15T07:33:09.579975Z,614.89
1010,2025-01-30,West,108,Tablet,3,415.43,2025,1,2026-03-15T07:33:09.579975Z,1246.29


In [0]:
# Writing cleaned dataset to Silver layer
# Data is partitioned by year and month for efficient querying
silver_df.write \
.mode("overwrite") \
.partitionBy("year","month") \
.parquet("abfss://silver@smartgearstorage428.dfs.core.windows.net/smartgear_sales/")

In [0]:
dbutils.fs.ls("abfss://silver@smartgearstorage428.dfs.core.windows.net/")

[FileInfo(path='abfss://silver@smartgearstorage428.dfs.core.windows.net/smartgear_sales/', name='smartgear_sales/', size=0, modificationTime=1773560108000)]

In [0]:
# Product-level summary
# Calculate total revenue and quantity sold for each product

product_summary = silver_df.groupBy("Product") \
.agg({"Revenue":"sum", "Quantity":"sum"}) \
.withColumnRenamed("sum(Revenue)", "TotalRevenue") \
.withColumnRenamed("sum(Quantity)", "TotalQuantity")

display(product_summary)

Product,TotalQuantity,TotalRevenue
Smartwatch,257,63432.83999999999
Laptop,259,207147.61000000013
Camera,337,168943.84000000005
Tablet,379,152735.87999999998
Drone,258,179999.84
Printer,298,44713.979999999996
Smartphone,301,181489.5500000001
Monitor,314,62501.48
Headphones,306,24618.980000000003
Gaming Console,292,132722.27


In [0]:
# Regional summary
# Calculate total revenue and quantity sold for each region

from pyspark.sql.functions import sum

region_summary = silver_df.groupBy("Region") \
.agg(
    sum("Revenue").alias("TotalRevenue"),
    sum("Quantity").alias("TotalQuantity")
)

display(region_summary)

Region,TotalRevenue,TotalQuantity
South,313016.63999999984,740
East,277434.47,719
West,274303.2000000001,694
North,353551.95999999973,848


In [0]:
# Gold layer: Region-level KPI dataset
# Calculate total revenue, total orders, and average order value per region

from pyspark.sql.functions import sum, count, round

region_kpi = silver_df.groupBy("Region").agg(
    sum("Revenue").alias("TotalRevenue"),
    count("OrderID").alias("TotalOrders"),
    round(sum("Revenue")/count("OrderID"),2).alias("AvgOrderValue")
)

display(region_kpi)

Region,TotalRevenue,TotalOrders,AvgOrderValue
South,313016.63999999984,248,1262.16
East,277434.47,240,1155.98
West,274303.2000000001,232,1182.34
North,353551.95999999973,280,1262.69


In [0]:
# Gold layer dataset
# Identify top 5 products by total revenue

from pyspark.sql.functions import sum, desc

top_products = silver_df.groupBy("Product") \
.agg(sum("Revenue").alias("TotalRevenue")) \
.orderBy(desc("TotalRevenue")) \
.limit(5)

display(top_products)

Product,TotalRevenue
Laptop,207147.61000000013
Smartphone,181489.5500000001
Drone,179999.84
Camera,168943.84000000005
Tablet,152735.87999999998


In [0]:
# Gold layer dataset
# Store-level performance metrics including revenue, orders, and average order value

from pyspark.sql.functions import sum, count, round

store_performance = silver_df.groupBy("StoreID").agg(
    sum("Revenue").alias("TotalRevenue"),
    count("OrderID").alias("TotalOrders"),
    round(sum("Revenue")/count("OrderID"),2).alias("AvgOrderValue")
)

display(store_performance)

StoreID,TotalRevenue,TotalOrders,AvgOrderValue
101,55990.11999999998,47,1191.28
112,67601.26,56,1207.17
113,52177.27,43,1213.42
110,76987.44,53,1452.59
107,66649.59999999996,51,1306.85
120,50465.28999999999,41,1230.86
118,49129.06,39,1259.72
104,56110.62,55,1020.19
102,51018.43,48,1062.88
111,49561.32000000001,43,1152.59


In [0]:
# Writing curated datasets to Gold layer
# Gold layer stores business-ready datasets for analytics and reporting

region_kpi.write.mode("overwrite") \
.parquet("abfss://gold@smartgearstorage428.dfs.core.windows.net/region_kpi/")

top_products.write.mode("overwrite") \
.parquet("abfss://gold@smartgearstorage428.dfs.core.windows.net/top_products/")

store_performance.write.mode("overwrite") \
.parquet("abfss://gold@smartgearstorage428.dfs.core.windows.net/store_performance/")